In [ ]:
import pandas as pd
import numpy as np

CURRENT_YEAR = 2026

# ============================================================
# LOAD FILES
# ============================================================

df = pd.read_csv("../data/drivearabia_car_depreciation_valuation/original/kia.csv")
dep_df = pd.read_csv("../data/drivearabia_car_depreciation_valuation/annual_dep_rate.csv")

# ============================================================
# CLEAN DEPRECIATION DATA
# ============================================================

dep_df["make"] = dep_df["make"].astype(str).str.strip().str.upper()
dep_df["model"] = dep_df["model"].astype(str).str.strip().str.upper()

# ============================================================
# KIA MODEL MAPPING
# ============================================================

MODEL_MAPPING = {
    "kia-carnival": "CARNIVAL",
    "kia-carens": "CARENS",
    "kia-cerato": "CERATO",
    "kia-ev5": "EV5",
    "kia-ev6": "EV6",
    "kia-k5": "K5",
    "kia-mohave": "MOHAVE",
    "kia-niro": "NIRO",
    "kia-picanto": "PICANTO",
    "kia-seltos": "SELTOS",
    "kia-sorento": "SORENTO",
    "kia-soul": "SOUL",
    "kia-sportage": "SPORTAGE",
    "kia-stinger": "STINGER",
    "kia-telluride": "TELLURIDE",
}

df["make"] = "KIA"

df["model_name"] = (
    df["model_slug"]
    .map(MODEL_MAPPING)
    .astype(str)
    .str.upper()
)

# ============================================================
# BUILD LOOKUP
# ============================================================

dep_lookup = (
    dep_df.groupby(["make", "model"])["annual_dep_rate"]
    .mean()
    .reset_index()
)

# ============================================================
# MERGE DEPRECIATION RATE
# ============================================================

df = df.merge(
    dep_lookup,
    left_on=["make", "model_name"],
    right_on=["make", "model"],
    how="left"
)

# ============================================================
# CAR AGE
# ============================================================

df["car_age"] = CURRENT_YEAR - df["year"]
df["car_age"] = df["car_age"].clip(lower=0)

# ============================================================
# DEPRECIATED VALUE
# ============================================================

def calc_depreciated_value(row):

    rate = row["annual_dep_rate"]

    if pd.isna(rate):
        return np.nan

    age = row["car_age"]

    if age == 0:
        return round(row["price_avg_aed"], 0)

    return round(
        row["price_avg_aed"] * ((1 - rate) ** age),
        0
    )

df["depreciated_value"] = df.apply(
    calc_depreciated_value,
    axis=1
)

# ============================================================
# VALIDATION
# ============================================================

print("Total Rows:", len(df))
print("Matched Rates:", df["annual_dep_rate"].notna().sum())
print("Missing Rates:", df["annual_dep_rate"].isna().sum())

print("\nUnmatched Models:")
print(
    df[df["annual_dep_rate"].isna()]
    ["model_slug"]
    .drop_duplicates()
    .tolist()
)

# ============================================================
# SAVE
# ============================================================

df.to_csv(
    "../data/drivearabia_car_depreciation_valuation/depreciated/kia_dep.csv",
    index=False
)

print("\nSaved: ../data/drivearabia_car_depreciation_valuation/depreciated/kia_dep.csv")

Total Rows: 198
Matched Rates: 135
Missing Rates: 63

Unmatched Models:
['kia-ev9', 'kia-k3', 'kia-k8', 'kia-pegas', 'kia-rio', 'kia-rio-hatchback', 'kia-sonet', 'kia-tasman']

Saved: data/kia_dep.csv
